### Phase 1 - Graph construction pipeline for Dataset A

0. GENERATE PARQUET FILE

In [1]:
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

# ------------------------- CONFIG -------------------------
DATA_DIR = "GothamDataset2025/processed"          # folder with all CSV files
PARQUET_PATH = "combined_raw_with_windows.parquet" # output file
WINDOW_SECONDS = 60

# ------------------------- CHECK IF ALREADY EXISTS -------------------------
if os.path.exists(PARQUET_PATH):
    print(f"Parquet file already exists at '{PARQUET_PATH}'. Skipping creation.")
else:
    # Find all CSV files
    csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
    print(f"Found {len(csv_files)} CSV files in {DATA_DIR}")

    # Read each CSV, adding a 'source_file' column
    dtype_spec = {'tcp.flags': str, 'tcp.checksum': str, 'tcp.options': str}
    all_dfs = []
    for fpath in tqdm(csv_files, desc="Reading CSVs"):
        df = pd.read_csv(fpath, parse_dates=['frame.time'], dtype=dtype_spec, low_memory=False)
        df['source_file'] = Path(fpath).stem
        all_dfs.append(df)

    # Concatenate
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f"Total rows read: {len(df_all)}")

    # Create window_start
    df_all['window_start'] = df_all['frame.time'].dt.floor(f'{WINDOW_SECONDS}s')

    # ---- Fill missing ports (optional, but good practice) ----
    port_cols = ['tcp.srcport','tcp.dstport','udp.srcport','udp.dstport']
    for col in port_cols:
        if col in df_all.columns:
            df_all[col] = df_all[col].fillna(0).astype(int)

    # Save to Parquet (fast, compressed)
    df_all.to_parquet(PARQUET_PATH, index=False)
    print(f"Saved combined raw data with window_start to '{PARQUET_PATH}'")
    print(f"File size: {os.path.getsize(PARQUET_PATH) / 1e9:.2f} GB")

Parquet file already exists at 'combined_raw_with_windows.parquet'. Skipping creation.


1. IMPORTS & CONFIGURATION

In [2]:
import os
import gc
import pickle
import json
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
CONFIG = {
    "parquet_path": "combined_raw_with_windows.parquet",
    "output_dir": "GothamDataset2025/processed_features/gotham_graphs",
    "window_seconds": 60,          # window length
    "stride_seconds": 30,          # stride (overlap) – new
    "min_packets_per_flow": 1,
    "sample_windows": None,        # None = all, or integer for quick test
    "seed": 42,
    "log_transform": True,
    "log_node_features": False,
    "clip_percentile": 99.0,
    "edge_features": [
        "packet_count", "total_bytes", "avg_size", "std_size", "min_size", "max_size",
        "duration", "packets_per_sec", "bytes_per_sec",
        "mean_iat", "std_iat", "min_iat", "max_iat",
        "sport", "dport",
        "syn", "ack", "rst", "fin",
        "mean_ttl", "min_ttl", "max_ttl",
        "mean_window", "min_window", "max_window",
        "entropy", "handshake_ratio", "rst_ratio"
    ],
    "node_features": [
        "total_flows", "total_packets", "total_bytes",
        "unique_dst_ips", "unique_src_ips",
        "unique_src_ports", "unique_dst_ports",
        "proto_diversity", "flow_balance",
        "mean_pkt_size_sent", "mean_pkt_size_recv",
        "max_flow_duration_sent", "max_flow_duration_recv"
    ],
    "protocol_categories": [1, 6, 17, 47]
}

OUTPUT_DIR = CONFIG["output_dir"]
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

2. HELPER FUNCTIONS (aggregation, node features, graph building)

In [3]:
def safe_groupby(df, by, **kwargs):
    try:
        return df.groupby(by, engine='pyarrow', **kwargs)
    except TypeError:
        return df.groupby(by, **kwargs)

def hex_to_int(s):
    if isinstance(s, str) and s.startswith('0x'):
        return int(s, 16)
    return 0

def entropy_per_flow(group):
    probs = group['count'] / group['count'].sum()
    return -np.sum(probs * np.log(probs + 1e-8))

def aggregate_window(df_win):
    """
    Given a DataFrame for a single window (already prepared with sport, dport, tcp_flags_int),
    aggregate flows and return a DataFrame of flows for that window.
    """
    if 'tcp.flags' in df_win.columns:
        df_win['tcp.flags'] = df_win['tcp.flags'].fillna('0x0')
        df_win['tcp_flags_int'] = df_win['tcp.flags'].apply(hex_to_int)
    else:
        df_win['tcp_flags_int'] = 0

    df_win = df_win.sort_values(['flow_key', 'frame.time'])
    df_win['iat'] = df_win.groupby('flow_key', sort=False)['frame.time'].diff().dt.total_seconds()

    agg_dict = {
        'frame.len': ['count', 'sum', 'min', 'max', 'mean', 'std'],
        'frame.time': ['min', 'max'],
        'ip.proto': 'first',
        'sport': 'first',
        'dport': 'first',
        'label': lambda x: 'Attack' if (x != 'Benign').any() else 'Benign',
        'iat': ['mean', 'std', 'min', 'max'],
        'ip.ttl': ['mean', 'min', 'max'],
        'tcp.window_size_value': ['mean', 'min', 'max']
    }

    grouped = safe_groupby(df_win, 'flow_key', sort=False)
    flows = grouped.agg(agg_dict).reset_index()
    flows.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in flows.columns.values]

    flows.rename(columns={
        'flow_key_': 'flow_key',
        'frame.len_count': 'packet_count',
        'frame.len_sum': 'total_bytes',
        'frame.len_min': 'min_size',
        'frame.len_max': 'max_size',
        'frame.len_mean': 'avg_size',
        'frame.len_std': 'std_size',
        'frame.time_min': 'first_time',
        'frame.time_max': 'last_time',
        'ip.proto_first': 'proto',
        'sport_first': 'sport',
        'dport_first': 'dport',
        'iat_mean': 'mean_iat',
        'iat_std': 'std_iat',
        'iat_min': 'min_iat',
        'iat_max': 'max_iat',
        'ip.ttl_mean': 'mean_ttl',
        'ip.ttl_min': 'min_ttl',
        'ip.ttl_max': 'max_ttl',
        'tcp.window_size_value_mean': 'mean_window',
        'tcp.window_size_value_min': 'min_window',
        'tcp.window_size_value_max': 'max_window',
    }, inplace=True)

    attack_name_series = (
        df_win.groupby('flow_key')['label']
        .agg(lambda x: x[x != 'Benign'].value_counts().index[0] if (x != 'Benign').any() else 'Benign')
        .reset_index(name='attack_name')
    )
    flows = flows.merge(attack_name_series, on='flow_key', how='left')

    flag_agg = df_win.groupby('flow_key')['tcp_flags_int'].agg(
        syn=lambda x: ((x & 0x02) > 0).sum(),
        ack=lambda x: ((x & 0x10) > 0).sum(),
        rst=lambda x: ((x & 0x04) > 0).sum(),
        fin=lambda x: ((x & 0x01) > 0).sum()
    ).reset_index()
    flows = flows.merge(flag_agg, on='flow_key', how='left').fillna(0)

    split_keys = flows['flow_key'].str.split('|', expand=True)
    flows['src_ip'] = split_keys[0]
    flows['dst_ip'] = split_keys[1]

    flows['duration'] = (flows['last_time'] - flows['first_time']).dt.total_seconds()
    flows['packets_per_sec'] = flows['packet_count'] / flows['duration'].replace(0, np.nan)
    flows['bytes_per_sec'] = flows['total_bytes'] / flows['duration'].replace(0, np.nan)

    size_counts = df_win.groupby(['flow_key', 'frame.len']).size().reset_index(name='count')
    entropy_series = size_counts.groupby('flow_key').apply(entropy_per_flow).reset_index(name='entropy')
    flows = flows.merge(entropy_series, on='flow_key', how='left')
    flows['entropy'] = flows['entropy'].fillna(0)

    eps = 1e-6
    flows['handshake_ratio'] = flows['syn'] / (flows['syn'] + flows['ack'] + eps)
    flows['rst_ratio'] = flows['rst'] / (flows['syn'] + flows['ack'] + flows['rst'] + flows['fin'] + eps)

    for col in ['mean_window', 'min_window', 'max_window']:
        flows[col] = flows[col].fillna(0)
    flows['std_size'] = flows['std_size'].fillna(0)
    flows['std_iat'] = flows['std_iat'].fillna(0)
    for col in ['packets_per_sec', 'bytes_per_sec']:
        flows[col] = flows[col].replace([np.inf, -np.inf], np.nan).fillna(0)
    for col in ['mean_ttl', 'min_ttl', 'max_ttl']:
        flows[col] = flows[col].fillna(64)

    flows.drop(columns=['first_time', 'last_time', 'flow_key'], inplace=True, errors='ignore')
    if CONFIG["min_packets_per_flow"] > 1:
        flows = flows[flows['packet_count'] >= CONFIG["min_packets_per_flow"]]

    return flows

def compute_node_features(flows_df):
    src_stats = flows_df.groupby('src_ip').agg(
        total_flows_sent=('packet_count', 'count'),
        total_packets_sent=('packet_count', 'sum'),
        total_bytes_sent=('total_bytes', 'sum'),
        unique_dst_ips=('dst_ip', 'nunique'),
        unique_src_ports=('sport', 'nunique'),
        unique_dst_ports=('dport', 'nunique'),
        proto_diversity=('proto', 'nunique'),
        mean_pkt_size_sent=('avg_size', 'mean'),
        max_flow_duration_sent=('duration', 'max')
    ).reset_index().rename(columns={'src_ip': 'ip'})

    dst_stats = flows_df.groupby('dst_ip').agg(
        total_flows_received=('packet_count', 'count'),
        total_packets_received=('packet_count', 'sum'),
        total_bytes_received=('total_bytes', 'sum'),
        unique_src_ips=('src_ip', 'nunique'),
        mean_pkt_size_recv=('avg_size', 'mean'),
        max_flow_duration_recv=('duration', 'max')
    ).reset_index().rename(columns={'dst_ip': 'ip'})

    node_feats = src_stats.merge(dst_stats, on='ip', how='outer').fillna(0)

    node_feats['total_flows'] = node_feats['total_flows_sent'] + node_feats['total_flows_received']
    node_feats['total_packets'] = node_feats['total_packets_sent'] + node_feats['total_packets_received']
    node_feats['total_bytes'] = node_feats['total_bytes_sent'] + node_feats['total_bytes_received']
    node_feats['flow_balance'] = node_feats['total_flows_sent'] / (node_feats['total_flows_received'] + 1e-6)
    node_feats['flow_balance'] = np.clip(node_feats['flow_balance'], None, 1000)

    if CONFIG.get("log_node_features", False):
        node_log_cols = ['total_bytes', 'total_packets', 'total_flows',
                         'mean_pkt_size_sent', 'mean_pkt_size_recv',
                         'max_flow_duration_sent', 'max_flow_duration_recv']
        for col in node_log_cols:
            if col in node_feats.columns:
                node_feats[col] = np.log1p(node_feats[col])

    for col in CONFIG["node_features"]:
        if col not in node_feats.columns:
            node_feats[col] = 0

    return node_feats[['ip'] + CONFIG["node_features"]]

def build_graph(flows_df, node_feats, window_start, edge_caps):
    if len(flows_df) == 0:
        return None

    all_ips = list(set(flows_df['src_ip'].unique()).union(set(flows_df['dst_ip'].unique())))
    if len(all_ips) == 0:
        return None
    ip_to_idx = {ip: i for i, ip in enumerate(all_ips)}

    x = []
    for ip in all_ips:
        row = node_feats[node_feats['ip'] == ip]
        if len(row) > 0:
            feats = row[CONFIG["node_features"]].values.flatten()
        else:
            feats = np.zeros(len(CONFIG["node_features"]))
        x.append(feats)
    x = np.array(x, dtype=np.float32)
    x = torch.tensor(x, dtype=torch.float)

    src_indices = flows_df['src_ip'].map(ip_to_idx).values
    dst_indices = flows_df['dst_ip'].map(ip_to_idx).values
    edge_index = torch.tensor([src_indices, dst_indices], dtype=torch.long)

    edge_data = flows_df[CONFIG["edge_features"]].copy()
    for col in edge_data.columns:
        cap = edge_caps.get(col)
        if cap is not None and cap > 0:
            edge_data[col] = np.clip(edge_data[col], None, cap)

    log_cols = ['total_bytes', 'packet_count', 'duration', 'mean_iat', 'std_iat',
                'packets_per_sec', 'bytes_per_sec', 'syn', 'ack', 'rst', 'fin',
                'avg_size', 'min_size', 'max_size', 'mean_ttl', 'min_ttl', 'max_ttl']
    if CONFIG.get("log_transform", False):
        for col in log_cols:
            if col in edge_data.columns:
                edge_data[col] = np.log1p(edge_data[col])

    continuous_attr = torch.tensor(edge_data.values, dtype=torch.float)

    proto_col = flows_df['proto'].values
    proto_onehot = np.zeros((len(proto_col), len(CONFIG["protocol_categories"]) + 1), dtype=np.float32)
    for i, p in enumerate(proto_col):
        if p in CONFIG["protocol_categories"]:
            idx = CONFIG["protocol_categories"].index(p)
            proto_onehot[i, idx] = 1.0
        else:
            proto_onehot[i, -1] = 1.0
    proto_tensor = torch.tensor(proto_onehot, dtype=torch.float)

    edge_attr = torch.cat([continuous_attr, proto_tensor], dim=1)

    if 'label' not in flows_df.columns:
        label_cols = [col for col in flows_df.columns if 'label' in col.lower()]
        if label_cols:
            flows_df = flows_df.rename(columns={label_cols[0]: 'label'})
        else:
            raise KeyError("No 'label' column found in flows_df")

    edge_y = torch.tensor((flows_df['label'] != 'Benign').astype(int).values, dtype=torch.long)
    y = torch.tensor([1 if (flows_df['label'] != 'Benign').any() else 0], dtype=torch.long)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, edge_y=edge_y)
    data.window_start = window_start
    if 'attack_name' in flows_df.columns:
        data.edge_attack_names = flows_df['attack_name'].tolist()
    else:
        data.edge_attack_names = flows_df['label'].tolist()
    return data

3. LOAD WINDOWS AND SPLIT

In [4]:
if not os.path.exists(CONFIG["parquet_path"]):
    raise FileNotFoundError(f"Parquet file not found: {CONFIG['parquet_path']}")

print("Generating sliding windows from Parquet...")

# Read min and max time to create window starts
time_series = pd.read_parquet(CONFIG["parquet_path"], columns=['frame.time'])
min_time = time_series['frame.time'].min()
max_time = time_series['frame.time'].max()

win_sec = CONFIG["window_seconds"]
stride_sec = CONFIG["stride_seconds"]

# Align start to a multiple of stride
start_ts = min_time.floor(f'{stride_sec}s')
window_starts = pd.date_range(start=start_ts, end=max_time, freq=f'{stride_sec}s')
window_starts = [w for w in window_starts if w <= max_time]
all_windows = list(window_starts)
print(f"Total potential windows: {len(all_windows)}")

# ---- Fast detection of non‑empty windows ----
print("Fast scanning for non‑empty windows...")
df_time = pd.read_parquet(CONFIG["parquet_path"], columns=['frame.time'])
df_time['frame.time'] = pd.to_datetime(df_time['frame.time'])

t = df_time['frame.time']
diff_sec = (t - start_ts).dt.total_seconds()
window_index = (diff_sec // stride_sec).astype(int)
window_starts_fast = start_ts + pd.to_timedelta(window_index * stride_sec, unit='s')

non_empty_windows = sorted(window_starts_fast.unique())
print(f"Non‑empty windows: {len(non_empty_windows)} / {len(all_windows)}")

# ---- USE ONLY NON-EMPTY WINDOWS ----
all_windows = non_empty_windows

if CONFIG["sample_windows"] is not None:
    all_windows = all_windows[:CONFIG["sample_windows"]]
    print(f"Sampled to {len(all_windows)} windows")

# ---- Temporal split (70% train, 15% val, 15% test) ----
n = len(all_windows)
train_end_idx = int(0.70 * n)
val_end_idx = int(0.85 * n)
train_windows = set(all_windows[:train_end_idx])
val_windows = set(all_windows[train_end_idx:val_end_idx])
test_windows = set(all_windows[val_end_idx:])

print(f"Train windows: {len(train_windows)}, Val: {len(val_windows)}, Test: {len(test_windows)}")

Generating sliding windows from Parquet...
Total potential windows: 31814
Fast scanning for non‑empty windows...
Non‑empty windows: 1332 / 31814
Train windows: 932, Val: 200, Test: 200


4. PASS 1: COMPUTE GLOBAL CLIPPING CAPS (with checkpoint)

In [5]:
edge_caps_path = f"{OUTPUT_DIR}/edge_caps.pkl"
if os.path.exists(edge_caps_path):
    print("Loading edge caps from checkpoint...")
    with open(edge_caps_path, 'rb') as f:
        edge_caps = pickle.load(f)
else:
    print("Pass 1: Computing global clipping caps from training windows...")
    edge_caps = {col: None for col in CONFIG["edge_features"]}
    train_values = {col: [] for col in CONFIG["edge_features"]}
    win_sec = CONFIG["window_seconds"]

    for win in tqdm(train_windows, desc="Training windows"):
        # ---- Use Parquet filters for speed ----
        df_win = pd.read_parquet(
            CONFIG["parquet_path"],
            filters=[('frame.time', '>=', win), ('frame.time', '<', win + pd.Timedelta(seconds=win_sec))]
        )
        if len(df_win) == 0:
            continue
        # ---- Original aggregation starts here ----
        df_win['sport'] = 0
        df_win['dport'] = 0
        tcp_mask = df_win['ip.proto'] == 6
        udp_mask = df_win['ip.proto'] == 17
        df_win.loc[tcp_mask, 'sport'] = df_win.loc[tcp_mask, 'tcp.srcport'].fillna(0).astype(int)
        df_win.loc[tcp_mask, 'dport'] = df_win.loc[tcp_mask, 'tcp.dstport'].fillna(0).astype(int)
        df_win.loc[udp_mask, 'sport'] = df_win.loc[udp_mask, 'udp.srcport'].fillna(0).astype(int)
        df_win.loc[udp_mask, 'dport'] = df_win.loc[udp_mask, 'udp.dstport'].fillna(0).astype(int)
        df_win[['sport', 'dport']] = df_win[['sport', 'dport']].fillna(0).astype(int)
        df_win['flow_key'] = (df_win['ip.src'].astype(str) + '|' + df_win['ip.dst'].astype(str) + '|' +
                              df_win['ip.proto'].astype(str) + '|' + df_win['sport'].astype(str) + '|' +
                              df_win['dport'].astype(str))
        flows_win = aggregate_window(df_win)
        if len(flows_win) == 0:
            continue
        for col in CONFIG["edge_features"]:
            if col in flows_win.columns:
                train_values[col].extend(flows_win[col].dropna().values)

    for col in CONFIG["edge_features"]:
        if len(train_values[col]) > 0:
            vals = np.array(train_values[col])
            cap = np.percentile(vals, CONFIG["clip_percentile"])
            edge_caps[col] = cap if cap > 0 else 0.0

    with open(edge_caps_path, 'wb') as f:
        pickle.dump(edge_caps, f)
    del train_values
    gc.collect()
    print("Clipping caps computed and saved.")

Loading edge caps from checkpoint...


5. PASS 2: BUILD GRAPHS FOR ALL WINDOWS (with checkpoint)

In [6]:
import os
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

train_pkl = f"{OUTPUT_DIR}/train_graphs.pkl"
val_pkl   = f"{OUTPUT_DIR}/val_graphs.pkl"
test_pkl  = f"{OUTPUT_DIR}/test_graphs.pkl"

if os.path.exists(train_pkl) and os.path.exists(val_pkl) and os.path.exists(test_pkl):
    print("Loading final graph splits from checkpoints...")
    with open(train_pkl, 'rb') as f: train_graphs = pickle.load(f)
    with open(val_pkl, 'rb') as f: val_graphs = pickle.load(f)
    with open(test_pkl, 'rb') as f: test_graphs = pickle.load(f)
else:
    checkpoint_path = f"{OUTPUT_DIR}/graphs_checkpoint.pkl"
    if os.path.exists(checkpoint_path):
        print("Loading graphs checkpoint...")
        with open(checkpoint_path, 'rb') as f:
            ckpt = pickle.load(f)
        train_graphs = ckpt.get('train_graphs', [])
        val_graphs = ckpt.get('val_graphs', [])
        test_graphs = ckpt.get('test_graphs', [])
        last_idx = ckpt.get('last_idx', -1)
    else:
        print("Starting fresh graph building...")
        train_graphs, val_graphs, test_graphs = [], [], []
        last_idx = -1

    start_idx = last_idx + 1
    remaining_count = len(all_windows) - start_idx

    if remaining_count > 0:
        print(f"Resuming from window {start_idx} of {len(all_windows)}...")
    else:
        print("All windows already processed.")

    win_sec = CONFIG["window_seconds"]

    # Only load data and process if there are remaining windows.
    if remaining_count > 0:
        cols_to_load = [
            'frame.time', 'frame.len', 'ip.proto', 'ip.src', 'ip.dst',
            'tcp.srcport', 'tcp.dstport', 'udp.srcport', 'udp.dstport',
            'tcp.flags', 'ip.ttl', 'tcp.window_size_value', 'label'
        ]
        print("Loading Parquet data once...")
        df_all = pd.read_parquet(CONFIG["parquet_path"], columns=cols_to_load)
        df_all['frame.time'] = pd.to_datetime(df_all['frame.time'])
        print(f"Loaded {len(df_all):,} rows.")

        # Sort once by time for fast searchsorted slicing.
        # If the data is already sorted, this is a no-op.
        if not df_all['frame.time'].is_monotonic_increasing:
            df_all = df_all.sort_values('frame.time', kind='mergesort').reset_index(drop=True)

        # ---- Precompute port and flow-key columns once ----
        proto = df_all['ip.proto'].to_numpy()
        is_tcp = (proto == 6)
        is_udp = (proto == 17)

        tcp_sport = df_all['tcp.srcport'].fillna(0).to_numpy(dtype=np.int64)
        udp_sport = df_all['udp.srcport'].fillna(0).to_numpy(dtype=np.int64)
        tcp_dport = df_all['tcp.dstport'].fillna(0).to_numpy(dtype=np.int64)
        udp_dport = df_all['udp.dstport'].fillna(0).to_numpy(dtype=np.int64)

        df_all['sport'] = np.select([is_tcp, is_udp], [tcp_sport, udp_sport], default=0)
        df_all['dport'] = np.select([is_tcp, is_udp], [tcp_dport, udp_dport], default=0)

        df_all['flow_key'] = (
            df_all['ip.src'].astype(str) + '|' +
            df_all['ip.dst'].astype(str) + '|' +
            df_all['ip.proto'].astype(str) + '|' +
            df_all['sport'].astype(str) + '|' +
            df_all['dport'].astype(str)
        )

        # ---- Precompute row positions for all windows using searchsorted ----
        time_ns = df_all['frame.time'].astype('int64').to_numpy()

        # Convert all window starts to nanosecond int64 (same unit as datetime64[ns])
        window_starts_ns = np.array(
            [pd.Timestamp(w).value for w in all_windows],
            dtype=np.int64
        )
        window_ends_ns = window_starts_ns + win_sec * 1_000_000_000

        start_positions = np.searchsorted(time_ns, window_starts_ns, side='left')
        end_positions = np.searchsorted(time_ns, window_ends_ns, side='left')

        train_windows_set = set(train_windows)
        val_windows_set = set(val_windows)

        for win_idx in tqdm(
            range(start_idx, len(all_windows)),
            desc="Building graphs",
            initial=start_idx,
            total=len(all_windows)
        ):
            try:
                start_pos = start_positions[win_idx]
                end_pos = end_positions[win_idx]

                if start_pos == end_pos:
                    continue

                df_win = df_all.iloc[start_pos:end_pos]

                flows_win = aggregate_window(df_win)
                if len(flows_win) == 0:
                    continue

                node_feats = compute_node_features(flows_win)
                graph = build_graph(flows_win, node_feats, all_windows[win_idx], edge_caps)

                if graph is not None:
                    win = all_windows[win_idx]
                    if win in train_windows_set:
                        train_graphs.append(graph)
                    elif win in val_windows_set:
                        val_graphs.append(graph)
                    else:
                        test_graphs.append(graph)

                if (win_idx + 1) % 5 == 0 or win_idx == len(all_windows) - 1:
                    with open(checkpoint_path, 'wb') as f:
                        pickle.dump({
                            'train_graphs': train_graphs,
                            'val_graphs': val_graphs,
                            'test_graphs': test_graphs,
                            'last_idx': win_idx
                        }, f)

            except Exception as e:
                print(f"Error processing window {all_windows[win_idx]}: {e}")
                with open(checkpoint_path, 'wb') as f:
                    pickle.dump({
                        'train_graphs': train_graphs,
                        'val_graphs': val_graphs,
                        'test_graphs': test_graphs,
                        'last_idx': win_idx
                    }, f)
                raise

    # Save final graph splits
    with open(train_pkl, 'wb') as f:
        pickle.dump(train_graphs, f)
    with open(val_pkl, 'wb') as f:
        pickle.dump(val_graphs, f)
    with open(test_pkl, 'wb') as f:
        pickle.dump(test_graphs, f)

    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    print("Final graph splits saved.")

Starting fresh graph building...
Resuming from window 0 of 1332...
Loading Parquet data once...


Loaded 35,134,281 rows.


Building graphs: 100%|██████████| 1332/1332 [4:50:53<00:00, 13.10s/it]    


Final graph splits saved.


6. SAVE FINAL SPLITS AND NORMALIZATION STATISTICS

In [8]:
if not os.path.exists(CONFIG["parquet_path"]):
    raise FileNotFoundError(f"Parquet file not found: {CONFIG['parquet_path']}")

print("Generating sliding windows from Parquet...")

# Read min and max time to create window starts
time_series = pd.read_parquet(CONFIG["parquet_path"], columns=['frame.time'])
min_time = time_series['frame.time'].min()
max_time = time_series['frame.time'].max()

win_sec = CONFIG["window_seconds"]
stride_sec = CONFIG["stride_seconds"]

# Align start to a multiple of stride
start_ts = min_time.floor(f'{stride_sec}s')
# Generate all window starts from start_ts to max_time
window_starts = pd.date_range(start=start_ts, end=max_time, freq=f'{stride_sec}s')
window_starts = [w for w in window_starts if w <= max_time]
all_windows = list(window_starts)
print(f"Total potential windows: {len(all_windows)}")

# ---- Fast detection of non‑empty windows ----
print("Fast scanning for non‑empty windows...")
# Read only frame.time column
df_time = pd.read_parquet(CONFIG["parquet_path"], columns=['frame.time'])
df_time['frame.time'] = pd.to_datetime(df_time['frame.time'])

t = df_time['frame.time']
diff_sec = (t - start_ts).dt.total_seconds()
window_index = (diff_sec // stride_sec).astype(int)
window_starts_fast = start_ts + pd.to_timedelta(window_index * stride_sec, unit='s')

# Get unique window starts that have at least one row
non_empty_windows = sorted(window_starts_fast.unique())
print(f"Non‑empty windows: {len(non_empty_windows)} / {len(all_windows)}")

# Use only non‑empty windows for the split
all_windows = non_empty_windows

if CONFIG["sample_windows"] is not None:
    all_windows = all_windows[:CONFIG["sample_windows"]]
    print(f"Sampled to {len(all_windows)} windows")

# ---- Temporal split (70% train, 15% val, 15% test) on non‑empty windows ----
n = len(all_windows)
train_end_idx = int(0.70 * n)
val_end_idx = int(0.85 * n)
train_windows = set(all_windows[:train_end_idx])
val_windows = set(all_windows[train_end_idx:val_end_idx])
test_windows = set(all_windows[val_end_idx:])

print(f"Train windows: {len(train_windows)}, Val: {len(val_windows)}, Test: {len(test_windows)}")

Generating sliding windows from Parquet...
Total potential windows: 31814
Fast scanning for non‑empty windows...
Non‑empty windows: 1332 / 31814
Train windows: 932, Val: 200, Test: 200
